# GRPO Text-to-SQL Fine-Tuning on Ray

Two-phase training for text-to-SQL on RHOAI:

1. **SFT Warmup** — LoRA SFT on cleaned BIRD-Platinum + NNDSS data to give the model baseline SQL ability
2. **GRPO RL** — Reinforcement learning with execution-based rewards against SQLite (BIRD) and Trino (NNDSS)

Both phases run as RayJobs on MIG 3g.71gb slices via CodeFlare SDK.

Based on the [ReViSQL](https://thinkingmachines.ai/news/putting-task-expertise-into-rl) approach.

## Setup

In [ ]:
%env UV_EXTRA_INDEX_URL=https://pypi.org/simple
!uv pip install codeflare_sdk matplotlib mlflow

In [ ]:
from codeflare_sdk import ManagedClusterConfig, RayJob
from kubernetes.client import (
    V1PersistentVolumeClaimVolumeSource,
    V1Volume,
    V1VolumeMount,
)

## Authenticate to your OpenShift Cluster

In [ ]:
!oc login --token=<YOUR_TOKEN> --server=<YOUR_API_SERVER> --insecure-skip-tls-verify
!oc whoami
!oc project
# Grant the workbench SA permission to create RayJobs
!oc apply -f ../manifests/rayjob-rbac.yaml
# Grant MLflow access to all SAs in this namespace (Ray creates dynamic SAs per job)
!oc create rolebinding mlflow-all-sa --clusterrole=mlflow-operator-mlflow-integration --group=system:serviceaccounts:$(oc project -q) 2>/dev/null || echo "mlflow-all-sa binding exists"

## 1. Configure Training Parameters

In [ ]:
import subprocess

# Cluster configuration
NAMESPACE = subprocess.run(
    ["oc", "project", "-q"], capture_output=True, text=True
).stdout.strip()
IMAGE = "quay.io/modh/ray@sha256:f63a015302758f805e5332605669b886e4d7ac60ec929413a2ffc19a904211c6"  # ray:2.55.1-py312-cu129-th081
HF_TOKEN = ""

# Kueue LocalQueue — set to "" to disable Kueue scheduling
KUEUE_LQ_NAME = "reserved"

# MIG GPU configuration (H200 slices)
GPU_RESOURCE_NAME = "nvidia.com/mig-3g.71gb"  # 71GB VRAM per slice
NUM_WORKERS = 1
N_GPUS_PER_NODE = 2  # 2x MIG slices

# Storage
PVC_NAME = "rl-sql-rwx"
PVC_PATH = "shared"
PVC_MOUNT_PATH = f"/opt/app-root/src/{PVC_PATH}"

# Model
MODEL_PATH = "Qwen/Qwen3-4B"

# Phase 1: SFT warmup paths (from notebook 01)
SFT_DATA_PATH = f"{PVC_MOUNT_PATH}/text2sql/sft_train_data.jsonl"
SFT_CKPT_DIR = f"{PVC_MOUNT_PATH}/text2sql/sft_checkpoint"

# Phase 2: GRPO paths
GRPO_DATA_PATH = f"{PVC_MOUNT_PATH}/text2sql/grpo_prompts.jsonl"
GRPO_OUTPUT_DIR = f"{PVC_MOUNT_PATH}/text2sql/grpo_output"

# SFT hyperparameters
SFT_EPOCHS = 2
SFT_MAX_SEQ_LEN = 4096
SFT_LEARNING_RATE = 1e-4

# GRPO hyperparameters
NUM_ITERATIONS = 15
TASKS_PER_ITERATION = 32
GROUP_SIZE = 16
N_TRAIN = 2500
N_VAL = 50
GRPO_LEARNING_RATE = 5e-6
MAX_PROMPT_LENGTH = 4096
MAX_RESPONSE_LENGTH = 512

# LoRA configuration (rank 32 per ReViSQL, alpha=r for 1.0 scaling)
LORA_R = 32
LORA_ALPHA = 32

# vLLM memory
GPU_MEMORY_UTILIZATION = 0.65

# MLflow tracking — operator-managed MLflow
MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
MLFLOW_EXPERIMENT_NAME = "rl-sql-grpo"

# Trino connection for NNDSS execution rewards
TRINO_HOST = f"trino.{NAMESPACE}.svc.cluster.local"
TRINO_PORT = 8080

TOTAL_GPUS = N_GPUS_PER_NODE * NUM_WORKERS
_agent_workers = TOTAL_GPUS * 4
_total_samples = TASKS_PER_ITERATION * GROUP_SIZE
assert _total_samples % _agent_workers == 0, (
    f"{TASKS_PER_ITERATION} * {GROUP_SIZE} = {_total_samples} not divisible by {_agent_workers}"
)

print(f"Namespace:  {NAMESPACE}")
print(f"Kueue LQ:   {KUEUE_LQ_NAME or '(disabled)'}")
print(f"GPU:        {NUM_WORKERS} worker x {N_GPUS_PER_NODE} {GPU_RESOURCE_NAME} = {TOTAL_GPUS} total ({TOTAL_GPUS * 71}GB)")
print(f"PVC:        {PVC_NAME}")
print(f"Model:      {MODEL_PATH}")
print(f"Batch:      {TASKS_PER_ITERATION} x {GROUP_SIZE} = {_total_samples} / {_agent_workers} workers")
print(f"vLLM mem:   {GPU_MEMORY_UTILIZATION}")
print(f"MLflow:     {MLFLOW_TRACKING_URI}")
print(f"Experiment: {MLFLOW_EXPERIMENT_NAME}")

## 2. Configure Shared Storage

In [ ]:
pvc_volume = V1Volume(
    name="training-data",
    persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=PVC_NAME),
)
pvc_mount = V1VolumeMount(name="training-data", mount_path=PVC_MOUNT_PATH)

---
## Phase 1: SFT Warmup

Train a LoRA adapter on the combined BIRD-Platinum + NNDSS SFT data.
This gives the model baseline SQL generation ability so GRPO groups
have meaningful variance in quality.

In [ ]:
env_vars = {}
if HF_TOKEN:
    env_vars["HF_TOKEN"] = HF_TOKEN
    env_vars["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
env_vars["TRINO_HOST"] = TRINO_HOST
env_vars["TRINO_PORT"] = str(TRINO_PORT)
env_vars["BIRD_DB_ROOT"] = f"{PVC_MOUNT_PATH}/text2sql/bird_databases"

# MLflow tracking — operator-managed MLflow (CR mode)
env_vars["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
env_vars["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
env_vars["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
env_vars["MLFLOW_WORKSPACE"] = NAMESPACE

# Read workbench SA token for MLflow Bearer auth
# verl's Tracking.__init__ runs on the head pod as a Ray actor, not our entrypoint,
# so MLFLOW_TRACKING_TOKEN must be a pod-level env var
try:
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as f:
        env_vars["MLFLOW_TRACKING_TOKEN"] = f.read().strip()
    print("MLflow SA token: read OK")
except FileNotFoundError:
    print("WARNING: No SA token found, MLflow auth will fail")

kueue_labels = {"kueue.x-k8s.io/queue-name": KUEUE_LQ_NAME} if KUEUE_LQ_NAME else {}

# Register MIG resource type with CodeFlare SDK
mig_accelerator_configs = {GPU_RESOURCE_NAME: GPU_RESOURCE_NAME}

sft_cluster_config = ManagedClusterConfig(
    image=IMAGE,
    num_workers=0,
    head_cpu_requests=4,
    head_cpu_limits=8,
    head_memory_requests=64,
    head_memory_limits=64,
    head_accelerators={GPU_RESOURCE_NAME: 1},
    volumes=[pvc_volume],
    volume_mounts=[pvc_mount],
    envs=env_vars,
    labels=kueue_labels,
    accelerator_configs=mig_accelerator_configs,
)

print(f"SFT cluster: head-only with 1x {GPU_RESOURCE_NAME}")
print(f"  Kueue: {KUEUE_LQ_NAME or '(disabled)'}")
print(f"  MLflow: {MLFLOW_TRACKING_URI} (workspace={NAMESPACE})")
print(f"  HF_TOKEN: {'set' if HF_TOKEN else 'not set'}")

In [ ]:
sft_entrypoint = f'''python -c "
import os, subprocess

# Ensure output dir is writable by Ray pod user
os.makedirs('{SFT_CKPT_DIR}', exist_ok=True)
os.chmod('{SFT_CKPT_DIR}', 0o777)

from training_hub import lora_sft

lora_sft(
    model_path='{MODEL_PATH}',
    data_path='{SFT_DATA_PATH}',
    ckpt_output_dir='{SFT_CKPT_DIR}',
    num_epochs={SFT_EPOCHS},
    max_seq_len={SFT_MAX_SEQ_LEN},
    learning_rate={SFT_LEARNING_RATE},
    lora_r={LORA_R},
    lora_alpha={LORA_ALPHA},
)

subprocess.run(['chmod', '-R', 'g+r', '{SFT_CKPT_DIR}'], check=True)
print('SFT warmup completed')
"'''

print("SFT entrypoint configured")
print(f"  Model: {MODEL_PATH}")
print(f"  Data:  {SFT_DATA_PATH}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")

In [ ]:
sft_job = RayJob(
    job_name="text2sql-sft-warmup",
    entrypoint=sft_entrypoint,
    cluster_config=sft_cluster_config,
    namespace=NAMESPACE,
    ttl_seconds_after_finished=600,
    local_queue=KUEUE_LQ_NAME or None,
)

# Pin submitter pod to GPU nodes so image is only pulled once
_orig_build = sft_job._build_rayjob_cr
def _patched_build():
    cr = _orig_build()
    tpl = cr["spec"].get("submitterPodTemplate")
    if tpl is None:
        cr["spec"]["submitterPodTemplate"] = {
            "spec": {
                "restartPolicy": "Never",
                "nodeSelector": {"nvidia.com/gpu.present": "true"},
                "containers": [{"name": "ray-job-submitter", "image": IMAGE}],
            }
        }
    else:
        tpl.setdefault("spec", {})["nodeSelector"] = {"nvidia.com/gpu.present": "true"}
        tpl["spec"].setdefault("restartPolicy", "Never")
    return cr
sft_job._build_rayjob_cr = _patched_build

sft_job.submit()
print(f"SFT RayJob '{sft_job.name}' submitted to namespace '{NAMESPACE}'")

In [ ]:
import time

print("Waiting for SFT warmup to complete...")
for i in range(120):
    status, ready = sft_job.status(print_to_console=False)
    print(f"[{i * 30}s] Status: {status.value}")
    if ready or status.value in ("FAILED", "COMPLETE"):
        break
    time.sleep(30)

print(f"\nSFT final status: {status.value}")

In [ ]:
sft_job.status()

---
## Phase 2: GRPO RL with Execution Rewards

Train with GRPO using the verl backend, starting from the SFT checkpoint.
The reward function executes generated SQL against SQLite (BIRD) and Trino (NNDSS)
and grades results using the ReViSQL grading logic.

Training Hub's `lora_grpo()` supports custom rewards via `reward_fn` parameter.
The data must have `messages` (prompt) and `ground_truth` (gold SQL) fields.

First, clean up the SFT workload so it releases its Kueue GPU quota.

In [ ]:
# Release Kueue quota from completed SFT job
!oc delete workload -l ray.io/rayjob-name=text2sql-sft-warmup --ignore-not-found
!oc delete rayjob text2sql-sft-warmup --ignore-not-found

In [ ]:
grpo_cluster_config = ManagedClusterConfig(
    image=IMAGE,
    head_cpu_requests=4,
    head_cpu_limits=8,
    head_memory_requests=64,
    head_memory_limits=64,
    num_workers=NUM_WORKERS,
    worker_cpu_requests=16,
    worker_cpu_limits=32,
    worker_memory_requests=256,
    worker_memory_limits=320,
    worker_accelerators={GPU_RESOURCE_NAME: N_GPUS_PER_NODE},
    volumes=[pvc_volume],
    volume_mounts=[pvc_mount],
    envs=env_vars,
    labels=kueue_labels,
    accelerator_configs=mig_accelerator_configs,
)

print(f"GRPO cluster: head (64GB) + {NUM_WORKERS} worker ({N_GPUS_PER_NODE}x {GPU_RESOURCE_NAME}, 32 CPU, 320GB RAM)")
print(f"  Kueue: {KUEUE_LQ_NAME or '(disabled)'}")

In [ ]:
import base64, glob, shutil, subprocess, zipfile

TOTAL_GPUS = N_GPUS_PER_NODE * NUM_WORKERS
REWARD_FN_PATH = f"{PVC_MOUNT_PATH}/text2sql/reward/verl_reward.py"
BIRD_DB_ROOT = f"{PVC_MOUNT_PATH}/text2sql/bird_databases"

# --- 1. Copy execution-based reward module to PVC ---
reward_src = os.path.join(os.path.dirname(os.getcwd()), "reward")
reward_dst = os.path.join(PVC_MOUNT_PATH, "text2sql", "reward")
os.makedirs(reward_dst, exist_ok=True)
for py_file in glob.glob(os.path.join(reward_src, "*.py")):
    shutil.copy2(py_file, reward_dst)
print(f"Reward module copied to {reward_dst}")

# Write verl-compatible adapter that wraps execution-based reward
_verl_adapter = '''\
import os, sys
sys.path.insert(0, os.path.dirname(__file__))
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

from reward_fn import compute_reward, extract_sql

def compute_score(data_source, solution_str, ground_truth, extra_info=None, **kwargs):
    """verl-compatible reward interface wrapping execution-based grading."""
    metadata = {"gold_sql": ground_truth, "db_type": "bird", "grading_method": "set"}
    if extra_info:
        if isinstance(extra_info, dict):
            metadata.update(extra_info)
        elif isinstance(extra_info, str):
            metadata["db_id"] = extra_info

    if not metadata.get("db_id"):
        sql = extract_sql(solution_str)
        if not sql:
            return -1.0
        from reward_fn import _THINK_PATTERN
        import re
        clean = _THINK_PATTERN.sub("", solution_str).strip() if solution_str else ""
        return 0.0

    return compute_reward(prompt="", response=solution_str, metadata=metadata)
'''
with open(os.path.join(reward_dst, "verl_reward.py"), "w") as f:
    f.write(_verl_adapter)
print(f"Verl reward adapter written to {REWARD_FN_PATH}")

# --- 2. Ensure BIRD databases are on PVC ---
db_count = len(glob.glob(os.path.join(BIRD_DB_ROOT, "*/*.sqlite")))
if db_count >= 11:
    print(f"BIRD databases present: {db_count} databases")
else:
    print(f"Only {db_count} BIRD databases found — downloading...")
    dl_dir = "/tmp/bird_dl"
    zip_path = "/tmp/minidev.zip"
    os.makedirs(dl_dir, exist_ok=True)
    subprocess.run(
        ["curl", "-L", "-o", zip_path,
         "https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip"],
        check=True,
    )
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dl_dir)
    src = os.path.join(dl_dir, "minidev", "MINIDEV", "dev_databases")
    os.makedirs(BIRD_DB_ROOT, exist_ok=True)
    from pathlib import Path
    for db_dir in Path(src).iterdir():
        dest = Path(BIRD_DB_ROOT) / db_dir.name
        if not dest.exists():
            shutil.copytree(db_dir, dest)
    shutil.rmtree(dl_dir, ignore_errors=True)
    os.remove(zip_path)
    db_count = len(glob.glob(os.path.join(BIRD_DB_ROOT, "*/*.sqlite")))
    print(f"Downloaded {db_count} BIRD databases")

# --- 3. Build GRPO entrypoint ---
# Start from SFT checkpoint (not base model) for better GRPO starting point
GRPO_MODEL_PATH = SFT_CKPT_DIR

_code = f"""\
import os, subprocess

os.makedirs('{GRPO_OUTPUT_DIR}', exist_ok=True)
os.chmod('{GRPO_OUTPUT_DIR}', 0o777)

# Set MLflow auth token from SA before any mlflow import
try:
    with open('/var/run/secrets/kubernetes.io/serviceaccount/token') as f:
        os.environ['MLFLOW_TRACKING_TOKEN'] = f.read().strip()
except FileNotFoundError:
    pass

subprocess.run(['pip', 'install', 'trino', 'mlflow'], check=True)

from training_hub import lora_grpo

result = lora_grpo(
    model_path='{GRPO_MODEL_PATH}',
    data_path='{GRPO_DATA_PATH}',
    ckpt_output_dir='{GRPO_OUTPUT_DIR}',
    reward_fn='{REWARD_FN_PATH}',
    backend='verl',
    n_gpus={TOTAL_GPUS},
    num_iterations={NUM_ITERATIONS},
    group_size={GROUP_SIZE},
    n_train={N_TRAIN},
    n_val={N_VAL},
    lora_r={LORA_R},
    lora_alpha={LORA_ALPHA},
    learning_rate={GRPO_LEARNING_RATE},
    gpu_memory_utilization={GPU_MEMORY_UTILIZATION},
    max_prompt_length={MAX_PROMPT_LENGTH},
    max_tokens={MAX_RESPONSE_LENGTH},
    mlflow_tracking_uri='{MLFLOW_TRACKING_URI}',
    mlflow_experiment_name='{MLFLOW_EXPERIMENT_NAME}',
    # Entropy/KL regularization to prevent collapse
    kl_loss_coef=0.001,
    entropy_coeff=0.001,
    clip_ratio=0.2,
)

subprocess.run(['chmod', '-R', 'g+r', '{GRPO_OUTPUT_DIR}'], check=True)

print('GRPO training complete')
print('Status:', result.get('status'))
print('Reward history:', result.get('reward_history'))
"""

_encoded = base64.b64encode(_code.encode()).decode()
grpo_entrypoint = (
    f"python -c \"import base64; exec(base64.b64decode('{_encoded}').decode())\""
)

print(f"\nGRPO entrypoint configured")
print(f"  Model: {GRPO_MODEL_PATH} (SFT checkpoint)")
print(f"  Reward: execution-based (SQLite + Trino)")
print(f"  GPUs: {TOTAL_GPUS} total")
print(f"  LR: {GRPO_LEARNING_RATE}, group_size: {GROUP_SIZE}")
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  KL coef: 0.001, entropy coef: 0.001")
print(f"  Iterations: {NUM_ITERATIONS}")

In [ ]:
grpo_job = RayJob(
    job_name="text2sql-grpo-training",
    entrypoint=grpo_entrypoint,
    cluster_config=grpo_cluster_config,
    namespace=NAMESPACE,
    ttl_seconds_after_finished=600,
    local_queue=KUEUE_LQ_NAME or None,
)

# Pin submitter pod to GPU nodes so image is only pulled once
_orig_build_grpo = grpo_job._build_rayjob_cr
def _patched_build_grpo():
    cr = _orig_build_grpo()
    tpl = cr["spec"].get("submitterPodTemplate")
    if tpl is None:
        cr["spec"]["submitterPodTemplate"] = {
            "spec": {
                "restartPolicy": "Never",
                "nodeSelector": {"nvidia.com/gpu.present": "true"},
                "containers": [{"name": "ray-job-submitter", "image": IMAGE}],
            }
        }
    else:
        tpl.setdefault("spec", {})["nodeSelector"] = {"nvidia.com/gpu.present": "true"}
        tpl["spec"].setdefault("restartPolicy", "Never")
    return cr
grpo_job._build_rayjob_cr = _patched_build_grpo

grpo_job.submit()
print(f"GRPO RayJob '{grpo_job.name}' submitted to namespace '{NAMESPACE}'")

In [ ]:
print("Waiting for GRPO training to complete...")
print(f"Expected duration: ~30-90 minutes for {NUM_ITERATIONS} iterations\n")

for i in range(240):
    status, ready = grpo_job.status(print_to_console=False)
    print(f"[{i * 30}s] Status: {status.value}")
    if ready or status.value in ("FAILED", "COMPLETE"):
        break
    time.sleep(30)

print(f"\nGRPO final status: {status.value}")

In [ ]:
grpo_job.status()

## 3. Plot Reward Curve

In [ ]:
import re
import matplotlib.pyplot as plt

# Read from PVC log (survives job cleanup) or fall back to live job logs
log_path = f"{PVC_MOUNT_PATH}/text2sql/grpo_output/verl_training.log"

try:
    with open(log_path) as f:
        log_output = f.read()
    print(f"Read training log from: {log_path}")
except FileNotFoundError:
    try:
        log_output = grpo_job.logs()
        print("Read from live job logs")
    except Exception:
        log_output = ""
        print(f"No logs available. Check {log_path} after training completes.")

# Extract reward scores from each step
scores = re.findall(r"critic/score/mean:([\d.]+)", log_output)

if scores:
    reward_values = [float(s) for s in scores]
    steps = list(range(1, len(reward_values) + 1))

    plt.figure(figsize=(10, 5))
    plt.plot(steps, reward_values, marker="o", markersize=4)
    plt.xlabel("Step")
    plt.ylabel("Mean Reward (SQL Match Score)")
    plt.title(f"GRPO Text-to-SQL — Reward Curve ({MODEL_PATH})")
    plt.grid(True, alpha=0.3)
    plt.axhline(y=reward_values[0], color='gray', linestyle='--', alpha=0.5, label=f"Initial: {reward_values[0]:.3f}")
    plt.axhline(y=reward_values[-1], color='red', linestyle='--', alpha=0.5, label=f"Latest: {reward_values[-1]:.3f}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Steps completed: {len(reward_values)}/180")
    print(f"Initial reward:  {reward_values[0]:.3f}")
    print(f"Latest reward:   {reward_values[-1]:.3f}")
    print(f"Best reward:     {max(reward_values):.3f} (step {reward_values.index(max(reward_values)) + 1})")
else:
    print("No reward scores found in logs yet.")

## 4. Cleanup

In [ ]:
# Clean up GRPO job and release Kueue GPU quota
!oc delete workload -l ray.io/rayjob-name=text2sql-grpo-training --ignore-not-found
!oc delete rayjob text2sql-grpo-training --ignore-not-found